<a href="https://colab.research.google.com/github/fernandodeeke/can2025/blob/main/gaussian_elimination2_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1> <h2></h2></h1></center>
<center><h1>Numerical Analysis</h1></center>
<center><h2>2026/2</h2></center>
<center><h3>Fernando Deeke Sasse</h3></center>
<center><h3>CCT - UDESC</h3></center>
<center><h2>Gaussian Elimination - 2</h2></center>

### 1. A different view of the Gaussian elimination method

As we have seen previously the Gaussian elimination method is the most widely used direct method for solving linear systems and is the basis for all variants of elimination methods. We will now look again at the two basic steps:

1. Elimination phase: we perform the elimination of the elements below the main diagonal of the augmented matrix $[A|B]$ by means of elementary row operations, until we obtain the echelon form.
2. Backsubstitution: we simply start solving recursively the resulting equations backwards.

We describe each of these steps below, now using elimination factors, better suited for the algorithmic implementation of the method.

### 2. Elimination phase
We call *pivot row* the row above which all elements below the main diagonal have already been zeroed.
In the expanded matrix, let the pivot row be $L_k$ and a typical line $L_i$ below to be transformed, in order to zero the element $a_{ik}$. To do so, we must multiply the pivot row by $\lambda = a_{ik}/a_{kk}$ and subtract the result from $L_i$, that is,

$$
L_i \rightarrow L_i - \lambda L_k\,.
$$

The row elements $L_i$ are updated as follows:

\begin{align}
&a_{ij} \leftarrow a_{ij} - \lambda a_{kj}\,\qquad j=k, \ldots , n\\
&b_i \leftarrow b_i - \lambda b_k\,.
\end{align}

As $a_{ik}=0$ by construction, to save computational time, it doesn't need to be calculated, so we can take $j=k+1, \ldots , n$.

The index $k$ is the index of the pivot row, so that $k=1, \ldots, n-1$ The index $i$ designates the row to be transformed, so that $i=k+1,\ \ldots, n$.

Remember that when using range(i,j), the actual range is $1, 2, \ldots, j-1$. For example,

In [ ]:
import numpy as np
c=np.array([1,2,3,4])
print(c[0])
print(c[-1])

In [ ]:
for j in range(0,3):
    print(c[j])

### 3. Gaussian elimination algorithm

The pseudocode is the following (supposing indices from 1 to n):

GaussianElimination(A, b)
    # A is the coefficient matrix (n × n)
    # b is the vector of independent terms (n × 1)
    n = length(b)
    FOR k = 1 TO n-1 DO          # k is the pivot row
        FOR i = k+1 TO n DO      # i is the row to be eliminated
            m = A[i,k] / A[k,k]  # calculate the multiplier
            FOR j = k+1 TO n DO  # update elements of row i
                A[i,j] = A[i,j] - m * A[k,j]
            END-FOR
            A[i,k] = 0           // set the element below the pivot to zero
            b[i] = b[i] - m * b[k]  // update the independent term
        END-FOR
    END-FOR
    RETURN [A|b]  # return the augmented matrix with A and b concatenated


The elimination algorithm can be performed in Python as shown below. Pay attention to the fact that the last element of the the matrix a has index n-1 and that range(0,n) goes from 0 to n-1.

In [ ]:
def GaussElimin(a,b):
    n = len(a)
    a = a.copy() # Preserve the input
    b = b.copy()
    for k in range(0,n-1): # define the index of the pivot element
        for i in range(k+1,n): # goes along the row below the pivot row and goes up to n-2 (last row)
            lam = a[i,k]/a[k,k]
            a[i,k+1:n] = a[i,k+1:n] - lam*a[k,k+1:n] # update the elements of row i
            a[i,k]=0
            b[i] = b[i] - lam*b[k]
    return  np.hstack([a,b])

Let us test the function:

In [5]:
A = np.array([[6.,1.,2.],[5.,11.,-3.],[-3.,4.,3.]])
print(A)

[[ 6.  1.  2.]
 [ 5. 11. -3.]
 [-3.  4.  3.]]


In [6]:
B = np.transpose(np.array([[2.,-4.,3.]]))
print(B)

[[ 2.]
 [-4.]
 [ 3.]]


Let's form the augmented matrix $M$:

In [7]:
M = np.hstack([A,B])
print(M)

[[ 6.  1.  2.  2.]
 [ 5. 11. -3. -4.]
 [-3.  4.  3.  3.]]


Perform the Gaussian elimination:

In [8]:
M1=GaussElimin(A,B)
print(M1)

[[ 6.          1.          2.          2.        ]
 [ 0.         10.16666667 -4.66666667 -5.66666667]
 [ 0.          0.          6.06557377  6.50819672]]


### 5. Backsubstitution algorithm
The following function takes as input a matrix of coefficients in echelon form and the column matrix of the homogeneous part. The output is the solution of the corresponding linear system.

In [10]:
def GaussRetro(a, b):
    a = a.copy() # Preserve the input
    b = b.copy()
    n = len(b)
    x = np.zeros(n)
    x[n-1] = (b[n-1] / a[n-1, n-1]).item()
    for k in range(n-2, -1, -1):
        x[k] = ((b[k] - np.dot(a[k, k+1:n], x[k+1:n])) / a[k, k]).item()
    return np.transpose([x])

To understand the use of the np.dot command above we should note that for each $k$, both $a[k,k+1:n]$ and $x[k+1:n]$ are computational vectors of length $n-k $. The np.dot command causes the respective components to be multiplied and added, just like in the usual dot product. The command range(n-1,-1,-1) causes the values of $k$ to start at $n-1$ and proceed downwards until reaching $k=0$ (which precedes -1). The .item() method extracts the scalar value of the quantity.

 Let's use the example above. Initially we slice the enlarged matrix. The matrix of coefficients is given by

In [12]:
A1 = M1[:,0:5]
A1

array([[ 6.        ,  1.        ,  2.        ,  2.        ],
       [ 0.        , 10.16666667, -4.66666667, -5.66666667],
       [ 0.        ,  0.        ,  6.06557377,  6.50819672]])

The non-homogeneous part is given by:

In [13]:
B1= M1[:,3]
B1

array([ 2.        , -5.66666667,  6.50819672])

In [15]:
B1v = np.reshape(B1,(3,1))
B1v

array([[ 2.        ],
       [-5.66666667],
       [ 6.50819672]])

Let us now apply the function that performs the backsubstitution:

In [16]:
X1 = GaussRetro(A1,B1v)
X1

array([[-0.01351351],
       [-0.06486486],
       [ 1.07297297]])

Let us verify the result:

In [17]:
A@X1-B

array([[ 0.0000000e+00],
       [ 0.0000000e+00],
       [-4.4408921e-16]])

We can solve linear systems with direct methods using numpy's solve command:

In [18]:
import numpy.linalg as la

In [19]:
x = la.solve(A,B)
print(x)

[[-0.01351351]
 [-0.06486486]
 [ 1.07297297]]


### 6.  Complete Gaussian elimination algorithm

Let's merge the two functions into one. The attribution a[i,k]=0 can now be removed because it is irrelevant:

In [20]:
import numpy as np

In [21]:
def GaussSolve(a, b):
    n = len(a)
    a = a.copy()  # Work with a copy to preserve the original matrix
    b = b.copy()  # Work with a copy to preserve the original vector
    x = np.zeros(n)
    for k in range(0, n-1):
        for i in range(k+1, n):
            lam = a[i, k] / a[k, k]
            a[i, k+1:n] = a[i, k+1:n] - lam * a[k, k+1:n]
            b[i] = b[i] - lam * b[k]
    x[n-1] = b[n-1] / a[n-1, n-1]
    for k in range(n-2, -1, -1):
        x[k] = (b[k] - np.dot(a[k, k+1:n], x[k+1:n])) / a[k, k]
    return x

Let's test the complete algorithm in the previous linear system.

In [22]:
A = np.array([[6., 1., 2., 4.], [5., 11., -3., 2.], [-3., 4., 3., 5.], [5., 2., 8., 3.]])
B = np.array([2., -4., 3., -7.])

In [23]:
X =  GaussSolve(A,B)
X

array([-0.32152756, -0.84200801, -1.1210348 ,  1.75331075])

Let us verify it:

In [24]:
residual = A @ X - B  # Compute the residual
residual

array([ 0.00000000e+00, -8.88178420e-16,  8.88178420e-16, -3.55271368e-15])

In [25]:
norm_inf = np.linalg.norm(residual, ord=np.inf)
print("\nInfinity Norm of Residual:")
print(norm_inf)


Infinity Norm of Residual:
3.552713678800501e-15


### 7. Example with large systems

Let's test larger random systems

In [32]:
np.random.seed(43453)
N = 10
A3 = np.random.rand(N,N)
B3 = np.random.rand(N,1)

Note that

In [33]:
A3

array([[0.7844481 , 0.76331947, 0.34735656, 0.68994884, 0.54723059,
        0.92579881, 0.8552435 , 0.10845538, 0.81883824, 0.22807891],
       [0.53062343, 0.48855489, 0.25874246, 0.92561893, 0.65422764,
        0.26436349, 0.78710372, 0.5741252 , 0.62885544, 0.36302539],
       [0.90102028, 0.07230066, 0.13481423, 0.10144185, 0.25571834,
        0.81142568, 0.12394265, 0.4700662 , 0.21006244, 0.10783463],
       [0.97646416, 0.0818504 , 0.7347487 , 0.76210741, 0.15670411,
        0.58941448, 0.60720567, 0.71621492, 0.33511445, 0.23142402],
       [0.8565556 , 0.38353716, 0.77656775, 0.941138  , 0.15197943,
        0.2975888 , 0.27419464, 0.98754726, 0.07571425, 0.32485921],
       [0.87860338, 0.94923304, 0.24366533, 0.00516716, 0.97191708,
        0.7843977 , 0.94401555, 0.54011364, 0.31077672, 0.85953907],
       [0.29203543, 0.06941725, 0.36115431, 0.985344  , 0.40196568,
        0.07404109, 0.22641815, 0.57320217, 0.71395281, 0.33665201],
       [0.43461381, 0.66330711, 0.6498346

In [34]:
X3 = GaussSolve(A3,B3)
X3

/tmp/ipykernel_792/2672761581.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x[n-1] = b[n-1] / a[n-1, n-1]
/tmp/ipykernel_792/2672761581.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x[k] = (b[k] - np.dot(a[k, k+1:n], x[k+1:n])) / a[k, k]


array([-0.78014194,  1.64411949, -1.13319994, -0.72833755, -2.02054919,
        1.06249775, -0.78987105,  2.26036594,  2.07989874, -0.4584725 ])

In [35]:
np.random.seed(43453)
N = 10
A3 = np.random.rand(N,N)

Let's check the answer by calculating the residue:

In [39]:
R = A3@X3-B3
R

array([[-4.44089210e-16, -5.94531152e-01,  9.38608456e-02,
        -7.68047790e-01, -3.62812224e-01, -6.00510028e-01,
        -3.19705226e-01, -7.12817477e-01, -3.42789164e-01,
         5.26543821e-02],
       [ 5.94531152e-01, -3.33066907e-16,  6.88391997e-01,
        -1.73516639e-01,  2.31718927e-01, -5.97887628e-03,
         2.74825926e-01, -1.18286326e-01,  2.51741988e-01,
         6.47185534e-01],
       [-9.38608456e-02, -6.88391997e-01, -5.10702591e-15,
        -8.61908636e-01, -4.56673070e-01, -6.94370874e-01,
        -4.13566071e-01, -8.06678323e-01, -4.36650010e-01,
        -4.12064635e-02],
       [ 7.68047790e-01,  1.73516639e-01,  8.61908636e-01,
        -7.32747196e-15,  4.05235566e-01,  1.67537762e-01,
         4.48342565e-01,  5.52303130e-02,  4.25258626e-01,
         8.20702172e-01],
       [ 3.62812224e-01, -2.31718927e-01,  4.56673070e-01,
        -4.05235566e-01, -2.22044605e-15, -2.37697804e-01,
         4.31069987e-02, -3.50005253e-01,  2.00230602e-02,
         4.

If the system is too large it is difficult to inspect all the components. In general, we evaluate the residue by calculating a norm of the corresponding vector. The most commonly used standard is the infinite norm, it is defined as being the highest magnitude among the components:  

$$
|\mathbf{u}|_{\infty}=\max_i\{{|u_i|},i=1,\ldots,n\}.
$$

In our case,  

In [40]:
import numpy.linalg as la

In [41]:
NR = la.norm(R, np.inf)
NR

np.float64(4.493306290091655)

Let's test the performance of the program we write with large systems.

In [42]:
np.random.seed(43453)
N = 300
A4 = np.random.rand(N,N)
B4 = np.random.rand(N)

In [43]:
X4 = GaussSolve(A4,B4)

Let's calculate the residue:

In [44]:
np.random.seed(43453)
N = 300
A4 = np.random.rand(N,N)
B4 = np.random.rand(N,1)

In [45]:
NR = la.norm(A4@X4-B4,np.inf)
NR

np.float64(151.04540487502643)

Let's estimate the CPU time required to solve this system:

In [46]:
np.random.seed(43453)
N = 300
A4 = np.random.rand(N,N)
B4 = np.random.rand(N)

In [47]:
%%timeit
GaussSolve(A4,B4)

280 ms ± 81.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


The CPU time required using numpy's solve function is given by:

In [48]:
np.random.seed(43453)
N = 300
A4 = np.random.rand(N,N)
B4 = np.random.rand(N)

In [49]:
%%timeit
la.solve(A,B)

9.26 µs ± 2.1 µs per loop (mean ± std. dev. of 7 runs, 100000 loops each)


As expected, the solver is much faster than our pedagogical algorithm, by almost four orders of magnitude.

### 8. Exercises

**1.** Solve step by step a random $4 \times 4$ linear system (use the first 4 digits of your cpf for the seed), describing each step of the GaussElimin and GaussRetro algorithm. Check the answer by calculating the residue.

**2.** Solve a random system $500 \times 500$ using the GaussSolve algorithm (do not show the result). Check the correctness of the result by calculating the residue norm and compare the required CPU time. Repeat the procedure using numpy's solver.  

**3** The **condition number** of a matrix $ A $, denoted by $ \kappa(A) $, measures the sensitivity of the solution of a linear system $ A \mathbf{x} = \mathbf{b} $ to such small perturbations in the matrix $ A $ or in the vector $ \mathbf{b} $. It is defined as:
$$
\kappa(A) = \|A\| \cdot \|A^{-1}\|,
$$
where $ \|A\| $ is the norm of the matrix (usually the 2-norm). A high condition number indicates that the system is ill-conditioned, i.e., small changes in the data can cause large variations in the solution. An example of an ill-conditioned matrix is ​​the Hilbert matrix $H_n$ of order $n$, defined by
$$
H_{ij} = \frac{1}{i + j + 1}, \quad i, j = 0, 1, \dots, n.
$$

We can define Python as follows:

In [ ]:
def hilb(n):
    return np.array([[1/(i+j+1) for i in range(n)] for j in range(n)])
    return np.array([[1/(i+j+1) for i in range(n)] for j in range(n)])

For example,

In [ ]:
hilb(4)

(i) Solve the linear system $HX=B$, where $H$ is the Hilbert matrix $H(30)$, $30 \times 30$ and B is a column matrix with all elements having the value 1. Use the Gaussian elimination algorithm developed earlier to solve the system. Calculate the residual in both cases.

(ii) Compare the condition number of $H$ with that of a random matrix $30 \times 30$. Use the command

(iii) Show that a small perturbation in one component of $H(30)$ causes the solution of the system defined in (i) to change drastically.


**4.** Study system instability with Hilbert matrices $n \times n$. For example, vary the value of a coefficient of B slightly, or add a small term to the matrix H. Use the solver in Numpy.

**5.** Define a system $GX=B$, $n \times n$, with $G=[g_{ij}]$ and $g_{ij}=1/(1+i+2j)$. Check if such a system is also unstable. Vary $n$. Use the solver in Numpy.

In [ ]:
import numpy as np
def m5(n):
    return np.array([[1/(i+2*j+1) for i in range(n)] for j in range(n)])

In [ ]:
m5(5)